In [21]:
import os
import glob
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import emoji
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import html
import requests
from datasets import Dataset, DatasetDict
from datetime import datetime, timedelta
from hmmlearn import hmm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


# load data

In [2]:
import os
import glob
import re
import pandas as pd

# Path ke folder berisi file CSV produk counterfeit
data_folder = "/Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/data"

output_folder = './cleaned_data'  # Folder untuk menyimpan hasil

# Buat folder output jika belum ada
os.makedirs(output_folder, exist_ok=True)

# Mendapatkan list semua file .csv
csv_files = glob.glob(os.path.join(data_folder, '*.csv'))

failed_files = []
success_files = []

for filepath in csv_files:
    filename = os.path.basename(filepath)
    
    # Regex untuk mengambil nama produk dari nama file
    match = re.search(r'(.+?)\.csv', filename)
    if match:
        product_id = match.group(1).strip()
    else:
        product_id = filename.replace('.csv', '').strip()
    
    print(f"\n{'='*60}")
    print(f"PROSES: {filename}")
    
    try:
        # BACA CSV
        df = pd.read_csv(filepath)
        print(f"  Kolom awal: {list(df.columns)}")
        print(f"  Jumlah baris awal: {len(df)}")
        
        # TAMBAH KOLOM product_id
        df['product_id'] = product_id
        
        # SESUAIKAN NAMA KOLOM REVIEW
        if 'review_text' in df.columns and 'review' not in df.columns:
            df.rename(columns={'review_text': 'review'}, inplace=True)
        
        # SESUAIKAN NAMA KOLOM TIMESTAMP
        if 'date' in df.columns and 'timestamp' not in df.columns:
            df.rename(columns={'date': 'timestamp'}, inplace=True)
        elif 'created_at' in df.columns and 'timestamp' not in df.columns:
            df.rename(columns={'created_at': 'timestamp'}, inplace=True)
        
        # TAMBAH KOLOM LABEL
        df['label'] = 1
        
        # === KOLOM YANG DISIMPAN ===
        cols_to_keep = ['product_id', 'review', 'timestamp', 'label']
        
        # Filter hanya kolom yang diinginkan
        cols_present = [c for c in cols_to_keep if c in df.columns]
        df = df[cols_present]
        
        # === DROP MISSING VALUES ===
        before_drop = len(df)
        
        # Drop missing values di kolom review
        if 'review' in df.columns:
            df = df.dropna(subset=['review'])
            df = df[df['review'].astype(str).str.strip() != '']
        
        # Drop missing values di kolom timestamp (jika ada)
        if 'timestamp' in df.columns:
            df = df.dropna(subset=['timestamp'])
        
        # Reset index
        df = df.reset_index(drop=True)
        
        after_drop = len(df)
        
        # === SIMPAN KE FILE CSV BARU ===
        output_filename = f"clean_{product_id}.csv"
        output_path = os.path.join(output_folder, output_filename)
        df.to_csv(output_path, index=False)
        
        success_files.append(output_filename)
        
        print(f"  ✓ BERHASIL!")
        print(f"  Kolom akhir: {list(df.columns)}")
        print(f"  Baris dihapus: {before_drop - after_drop}")
        print(f"  Baris tersimpan: {after_drop}")
        print(f"  Tersimpan ke: {output_filename}")
        
    except Exception as e:
        failed_files.append((filename, str(e)))
        print(f"  ✗ GAGAL: {str(e)}")



PROSES: Panda_Vivo Y71.csv
  Kolom awal: ['web_scraper_order', 'web_scraper_start_url', 'pagination', 'review', 'timestamp']
  Jumlah baris awal: 566
  ✓ BERHASIL!
  Kolom akhir: ['product_id', 'review', 'timestamp', 'label']
  Baris dihapus: 193
  Baris tersimpan: 373
  Tersimpan ke: clean_Panda_Vivo Y71.csv

PROSES: TAOB1688_Ip 17 Pro.csv
  Kolom awal: ['web_scraper_order', 'web_scraper_start_url', 'review', 'timestamp']
  Jumlah baris awal: 3
  ✓ BERHASIL!
  Kolom akhir: ['product_id', 'review', 'timestamp', 'label']
  Baris dihapus: 0
  Baris tersimpan: 3
  Tersimpan ke: clean_TAOB1688_Ip 17 Pro.csv

PROSES: PinDuoDuo Y17 Ram.csv
  Kolom awal: ['web_scraper_order', 'web_scraper_start_url', 'pagination', 'review', 'timestamp']
  Jumlah baris awal: 126
  ✓ BERHASIL!
  Kolom akhir: ['product_id', 'review', 'timestamp', 'label']
  Baris dihapus: 23
  Baris tersimpan: 103
  Tersimpan ke: clean_PinDuoDuo Y17 Ram.csv

PROSES: Temu Toko_Ip 17 Pro.csv
  Kolom awal: ['web_scraper_order', 'w

# PReprocessing text ulasan

In [18]:
# ============================================
# FUNGSI-FUNGSI CLEANING
# ============================================

def clean_text_for_bert(text):
    """Membersihkan text review untuk BERT"""
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus URL, mention (@), hashtag
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    
    # 3. Hapus HTML entity
    text = html.unescape(text)
    
    # 4. Normalisasi elongasi
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 5. Hapus emoji
    text = emoji.replace_emoji(text, replace='')
    
    # 6. Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def clean_timestamp(timestamp_str, reference_date=None):
    """Membersihkan dan menormalisasi berbagai format timestamp"""
    if pd.isna(timestamp_str) or timestamp_str == '' or timestamp_str is None:
        return None
    
    timestamp_str = str(timestamp_str).strip()
    
    if reference_date is None:
        reference_date = datetime.now()
    
    try:
        # Format 1: "2026-01-29 14:15 | Variasi: Biru Muda,8/256"
        if '|' in timestamp_str:
            timestamp_str = timestamp_str.split('|')[0].strip()
            try:
                return pd.to_datetime(timestamp_str).strftime('%Y-%m-%d %H:%M:%S')
            except:
                pass
        
        # Format 2: "X minggu lalu"
        minggu_pattern = r'(\d+)\s*minggu\s*(?:yang\s*)?lalu'
        match = re.search(minggu_pattern, timestamp_str, re.IGNORECASE)
        if match:
            weeks = int(match.group(1))
            calculated_date = reference_date - timedelta(weeks=weeks)
            return calculated_date.strftime('%Y-%m-%d %H:%M:%S')
        
        # Format 3: "X hari lalu"
        hari_pattern = r'(\d+)\s*hari\s*(?:yang\s*)?lalu'
        match = re.search(hari_pattern, timestamp_str, re.IGNORECASE)
        if match:
            days = int(match.group(1))
            calculated_date = reference_date - timedelta(days=days)
            return calculated_date.strftime('%Y-%m-%d %H:%M:%S')
        
        # Format 4: "X bulan lalu"
        bulan_pattern = r'(\d+)\s*bulan\s*(?:yang\s*)?lalu'
        match = re.search(bulan_pattern, timestamp_str, re.IGNORECASE)
        if match:
            months = int(match.group(1))
            calculated_date = reference_date - timedelta(days=months * 30)
            return calculated_date.strftime('%Y-%m-%d %H:%M:%S')
        
        # Format 5: "baru saja", "kemarin"
        if any(word in timestamp_str.lower() for word in ['baru saja', 'just now', 'sekarang']):
            return reference_date.strftime('%Y-%m-%d %H:%M:%S')
        
        if any(word in timestamp_str.lower() for word in ['kemarin', 'yesterday']):
            calculated_date = reference_date - timedelta(days=1)
            return calculated_date.strftime('%Y-%m-%d %H:%M:%S')
        
        # Format 6: Coba parse langsung
        try:
            return pd.to_datetime(timestamp_str).strftime('%Y-%m-%d %H:%M:%S')
        except:
            pass
        
        return timestamp_str
        
    except Exception as e:
        print(f"    ⚠ Error parsing timestamp '{timestamp_str}': {str(e)}")
        return timestamp_str


# ============================================
# PROSES UTAMA: GABUNGAN CLEANING TEXT & TIMESTAMP
# ============================================

# Path folder
input_folder = '/Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/cleaned_data'
output_folder = '/Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/bert_ready_data'

# Buat folder output jika belum ada
os.makedirs(output_folder, exist_ok=True)

# Set reference date untuk perhitungan timestamp relatif
# Sesuaikan dengan tanggal scraping data Anda
reference_date = datetime(2026, 7, 1)

# Cari semua file CSV
csv_files = glob.glob(os.path.join(input_folder, 'clean_*.csv'))

print(f"Reference date: {reference_date.strftime('%Y-%m-%d')}")
print(f"Input folder: {input_folder}")
print(f"Output folder: {output_folder}")
print(f"File CSV ditemukan: {len(csv_files)}")
print("="*60)

# Statistik keseluruhan
total_stats = {
    'files_processed': 0,
    'total_rows_before': 0,
    'total_rows_after': 0,
    'rows_removed_empty_review': 0,
    'rows_removed_invalid_timestamp': 0
}

for filepath in csv_files:
    filename = os.path.basename(filepath)
    print(f"\n{'='*60}")
    print(f"PROSES: {filename}")
    print(f"{'='*60}")
    
    try:
        # Baca CSV
        df = pd.read_csv(filepath)
        rows_before = len(df)
        print(f"  Baris awal: {rows_before}")
        print(f"  Kolom: {list(df.columns)}")
        
        # ============================================
        # 1. CLEANING TEXT REVIEW
        # ============================================
        if 'review' in df.columns:
            print(f"\n  [1] Membersihkan text review...")
            
            # Simpan contoh sebelum cleaning
            sample_before = str(df['review'].iloc[0][:100]) if len(df) > 0 else ""
            if sample_before:
                print(f"    Contoh sebelum: {sample_before}...")
            
            # Apply cleaning
            df['review'] = df['review'].apply(clean_text_for_bert)
            
            # Hapus review yang kosong setelah cleaning
            before_clean = len(df)
            df = df[df['review'].str.strip() != '']
            after_clean = len(df)
            removed_review = before_clean - after_clean
            
            # Contoh setelah cleaning
            if len(df) > 0:
                sample_after = str(df['review'].iloc[0][:100])
                print(f"    Contoh setelah: {sample_after}...")
            print(f"    Review kosong dihapus: {removed_review}")
            print(f"    Baris setelah cleaning text: {len(df)}")
        else:
            print(f"    ⚠ Kolom 'review' tidak ditemukan!")
            removed_review = 0
        
        # ============================================
        # 2. CLEANING TIMESTAMP
        # ============================================
        if 'timestamp' in df.columns:
            print(f"\n  [2] Membersihkan timestamp...")
            
            # Tampilkan contoh sebelum
            print(f"    Contoh timestamp sebelum:")
            for i, ts in enumerate(df['timestamp'].head(3)):
                print(f"      {i+1}. {ts}")
            
            # Clean timestamp
            df['timestamp'] = df['timestamp'].apply(
                lambda x: clean_timestamp(x, reference_date)
            )
            
            # Konversi ke datetime untuk validasi
            df['timestamp_parsed'] = pd.to_datetime(df['timestamp'], errors='coerce')
            
            # Hitung invalid
            invalid_count = df['timestamp_parsed'].isna().sum()
            
            # Hapus yang invalid
            before_ts = len(df)
            df = df.dropna(subset=['timestamp_parsed'])
            after_ts = len(df)
            removed_ts = before_ts - after_ts
            
            # Format final timestamp
            df['timestamp'] = df['timestamp_parsed'].dt.strftime('%Y-%m-%d %H:%M:%S')
            
            # Hapus kolom bantuan
            df = df.drop(columns=['timestamp_parsed'])
            
            # Tampilkan contoh setelah
            if len(df) > 0:
                print(f"    Contoh timestamp setelah:")
                for i, ts in enumerate(df['timestamp'].head(3)):
                    print(f"      {i+1}. {ts}")
            
            print(f"    Timestamp invalid dihapus: {removed_ts}")
            print(f"    Baris setelah cleaning timestamp: {len(df)}")
        else:
            print(f"\n  [2] ⚠ Kolom 'timestamp' tidak ditemukan!")
            removed_ts = 0
        
        # ============================================
        # 3. RESET INDEX & SIMPAN
        # ============================================
        df = df.reset_index(drop=True)
        rows_after = len(df)
        
        # Simpan ke folder output
        output_filename = f"bert_{filename}"
        output_path = os.path.join(output_folder, output_filename)
        df.to_csv(output_path, index=False)
        
        # Update statistik
        total_stats['files_processed'] += 1
        total_stats['total_rows_before'] += rows_before
        total_stats['total_rows_after'] += rows_after
        total_stats['rows_removed_empty_review'] += removed_review
        total_stats['rows_removed_invalid_timestamp'] += removed_ts
        
        # Ringkasan per file
        print(f"\n  {'='*40}")
        print(f"  RINGKASAN {filename}:")
        print(f"  Baris awal: {rows_before}")
        print(f"  Review kosong dihapus: {removed_review}")
        print(f"  Timestamp invalid dihapus: {removed_ts}")
        print(f"  Total dihapus: {removed_review + removed_ts}")
        print(f"  Baris akhir: {rows_after}")
        print(f"  ✓ Tersimpan: {output_filename}")
        
    except Exception as e:
        print(f"\n  ✗ GAGAL memproses {filename}: {str(e)}")
        import traceback
        traceback.print_exc()


Reference date: 2026-07-01
Input folder: /Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/cleaned_data
Output folder: /Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/bert_ready_data
File CSV ditemukan: 4

PROSES: clean_Temu Toko_Ip 17 Pro.csv
  Baris awal: 18
  Kolom: ['product_id', 'review', 'timestamp', 'label']

  [1] Membersihkan text review...
    Contoh sebelum: Keren ini mah hp nya kecil tapi gak kecil baget anak saya suka buat bocill cocok dari pada harus reb...
    Contoh setelah: keren ini mah hp nya kecil tapi gak kecil baget anak saya suka buat bocill cocok dari pada harus reb...
    Review kosong dihapus: 0
    Baris setelah cleaning text: 18

  [2] Membersihkan timestamp...
    Contoh timestamp sebelum:
      1. 2025-09-08 11:53 | Variasi: Silver
      2. 2025-09-08 11:57 | Variasi: Hitam
      3. 2025-09-18 17:02 | Variasi: Grey
    Contoh timestamp setelah:
      1. 2025-09-08 11:53:00
      2. 2025-09-08 11:57:00
      3. 2025-09-18 17:02:00
    Timestamp invalid dihapus:

# pelabelan sentiment menggunakan data training indobert

In [7]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pretrained= "mdhugol/indonesia-bert-sentiment-classification"

model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

pos_text = "Sangat bahagia hari ini"
neg_text = "Dasar anak sialan!! Kurang ajar!!"

result = sentiment_analysis(pos_text)
status = label_index[result[0]['label']]
score = result[0]['score']
print(f'Text: {pos_text} | Label : {status} ({score * 100:.3f}%)')

result = sentiment_analysis(neg_text)
status = label_index[result[0]['label']]
score = result[0]['score']
print(f'Text: {neg_text} | Label : {status} ({score * 100:.3f}%)')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: Sangat bahagia hari ini | Label : positive (99.481%)
Text: Dasar anak sialan!! Kurang ajar!! | Label : negative (99.828%)


### 5. Verifikasi Mapping Label (WAJIB)

Langkah ini bertujuan untuk memverifikasi secara empiris urutan label yang dikeluarkan oleh model IndoBERT yang digunakan, untuk mencegah terbaliknya interpretasi sentimen yang akan merusak probabilitas untuk HMM.

In [8]:
print("=== SECTION 5: VERIFIKASI MAPPING LABEL ===")

# Pastikan model ada di device yang tepat untuk mengatasi error MPS
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Menggunakan device: {device}")
model = model.to(device)
model.eval()

# 1. Siapkan kalimat uji
kalimat_uji = [
    {"kategori": "Jelas Positif", "teks": "Barangnya bagus banget, sesuai deskripsi, pengiriman cepat"},
    {"kategori": "Jelas Negatif", "teks": "Barang rusak, tidak sesuai foto, kecewa berat"},
    {"kategori": "Jelas Netral", "teks": "Barang sudah sampai, belum saya coba"}
]

print("\nHasil Uji Label Mentah (LABEL_0, LABEL_1, LABEL_2):")
for item in kalimat_uji:
    teks = item["teks"]
    
    # 2. Tokenisasi dan jalankan model, pastikan tensor di device yang sama dengan model
    inputs = tokenizer(teks, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().tolist()
        
    pred_idx = probs.index(max(probs))
    pred_label = f"LABEL_{pred_idx}"
    
    print(f"[{item['kategori']}] Teks: {teks}")
    print(f"-> Prediksi: {pred_label} (Confidence: {max(probs)*100:.2f}%)")
    print(f"-> Probabilitas: LABEL_0={probs[0]:.4f}, LABEL_1={probs[1]:.4f}, LABEL_2={probs[2]:.4f}\n")

# 3. Hasil verifikasi manual pada Section 5
# Silakan sesuaikan Dictionary di bawah ini berdasarkan hasil output di atas sebelum lanjut ke Section 6
label_map = {
    'LABEL_0': 'S_plus',
    'LABEL_1': 'S_netral',
    'LABEL_2': 'S_minus'
}

print(f"Label Map final yang digunakan untuk pipeline berikutnya: {label_map}")


=== SECTION 5: VERIFIKASI MAPPING LABEL ===
Menggunakan device: mps

Hasil Uji Label Mentah (LABEL_0, LABEL_1, LABEL_2):
[Jelas Positif] Teks: Barangnya bagus banget, sesuai deskripsi, pengiriman cepat
-> Prediksi: LABEL_0 (Confidence: 99.74%)
-> Probabilitas: LABEL_0=0.9974, LABEL_1=0.0018, LABEL_2=0.0008

[Jelas Negatif] Teks: Barang rusak, tidak sesuai foto, kecewa berat
-> Prediksi: LABEL_2 (Confidence: 99.81%)
-> Probabilitas: LABEL_0=0.0008, LABEL_1=0.0012, LABEL_2=0.9981

[Jelas Netral] Teks: Barang sudah sampai, belum saya coba
-> Prediksi: LABEL_1 (Confidence: 98.60%)
-> Probabilitas: LABEL_0=0.0018, LABEL_1=0.9860, LABEL_2=0.0123

Label Map final yang digunakan untuk pipeline berikutnya: {'LABEL_0': 'S_plus', 'LABEL_1': 'S_netral', 'LABEL_2': 'S_minus'}


### Preview Model pada Sampel Data Anda

Melihat output prediksi dari model IndoBERT pada beberapa sampel data asli Anda secara acak sebelum menjalankan batch inference secara keseluruhan.

In [9]:
print("=== PREVIEW PREDIKSI PADA SAMPEL DATA ASLI ===")

input_folder = './bert_ready_data'
csv_files = glob.glob(os.path.join(input_folder, 'bert_clean_*.csv'))

if not csv_files:
    print("Data tidak ditemukan di ./bert_ready_data")
else:
    # Ambil 1 file secara acak
    sample_file = random.choice(csv_files)
    df_sample = pd.read_csv(sample_file).dropna(subset=['review'])
    
    if not df_sample.empty:
        n_samples = min(5, len(df_sample))
        # Ambil sampel acak
        df_sample = df_sample.sample(n=n_samples, random_state=42)
        
        print(f"Mengambil {n_samples} sampel acak dari {os.path.basename(sample_file)}:\n")
        
        for idx, row in df_sample.iterrows():
            text = str(row['review'])
            
            # Inference sampel tunggal
            inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                # Output logits (angka mentah model)
                logits = outputs.logits.squeeze().tolist()
                # Output softmax probabilitas (yang dijumlahkan = 1)
                probs = F.softmax(outputs.logits, dim=-1).squeeze().tolist()
                
            pred_idx = probs.index(max(probs))
            
            # Pemetaan asumsi sementara (bisa dicocokkan dengan Section 5)
            label_names = {0: "Positif (LABEL_0)", 1: "Netral (LABEL_1)", 2: "Negatif (LABEL_2)"} 
            pred_name = label_names.get(pred_idx, f"LABEL_{pred_idx}")
            
            text_preview = (text[:150] + '...') if len(text) > 150 else text
            print(f"Review: \"{text_preview}\"")
            print(f"-> Prediksi Terkuat : {pred_name} (Confidence: {max(probs)*100:.2f}%)")
            print(f"-> Logits Mentah    : {logits}")
            print(f"-> Probabilitas     : Positif={probs[0]:.4f}, Netral={probs[1]:.4f}, Negatif={probs[2]:.4f}")
            print("-" * 60)


=== PREVIEW PREDIKSI PADA SAMPEL DATA ASLI ===
Mengambil 5 sampel acak dari bert_clean_Temu Toko_Ip 17 Pro.csv:

Review: "keren ini mah hp nya kecil tapi gak kecil baget anak saya suka buat bocill cocok dari pada harus rebutan sama saya harga terjangkau kualitas gak perlu..."
-> Prediksi Terkuat : Positif (LABEL_0) (Confidence: 99.78%)
-> Logits Mentah    : [4.624631881713867, -1.984306812286377, -2.40547251701355]
-> Probabilitas     : Positif=0.9978, Netral=0.0013, Negatif=0.0009
------------------------------------------------------------
Review: "di luar ekspetasi saya ga nyangka bakal se cute ini udah di coba berfungsi dengan baikk ga nyesel deh pokoknya respon penjual pun cepat sekali hitam d..."
-> Prediksi Terkuat : Positif (LABEL_0) (Confidence: 99.50%)
-> Logits Mentah    : [4.027828216552734, -2.2275338172912598, -1.7379337549209595]
-> Probabilitas     : Positif=0.9950, Netral=0.0019, Negatif=0.0031
------------------------------------------------------------
Review: "baran

### 6. Inference Batch (Strictly Per CSV File)

Mengeksekusi model satu per satu dari setiap file `bert_clean_*.csv` dan langsung menyimpannya ke folder output tanpa menggabungkan datanya sama sekali.

In [ ]:
# ============================================
# 6.1 SETUP DEVICE (CPU)
# ============================================
device = torch.device('cpu')
print(f"Using device: {device}")

# Pastikan model di CPU
model.to(device)
model.eval()

# ============================================
# 6.2 LABEL MAP (HASIL VERIFIKASI SECTION 5)
# ============================================
# GANTI SESUAI HASIL VERIFIKASI ANDA!
# Ini hanya contoh, sesuaikan dengan hasil Section 5
LABEL_MAP = {
    'LABEL_0': 'POSITIVE',   # <-- SESUAIKAN!
    'LABEL_1': 'NEUTRAL',    # <-- SESUAIKAN!
    'LABEL_2': 'NEGATIVE'    # <-- SESUAIKAN!
}

# Mapping indeks probabilitas ke sentimen
# prob[0] = LABEL_0, prob[1] = LABEL_1, prob[2] = LABEL_2
# SESUAIKAN urutan ini dengan hasil verifikasi!
SENTIMENT_ORDER = {
    0: 'S_plus' if LABEL_MAP['LABEL_0'] == 'POSITIVE' else 'S_netral' if LABEL_MAP['LABEL_0'] == 'NEUTRAL' else 'S_minus',
    1: 'S_plus' if LABEL_MAP['LABEL_1'] == 'POSITIVE' else 'S_netral' if LABEL_MAP['LABEL_1'] == 'NEUTRAL' else 'S_minus',
    2: 'S_plus' if LABEL_MAP['LABEL_2'] == 'POSITIVE' else 'S_netral' if LABEL_MAP['LABEL_2'] == 'NEUTRAL' else 'S_minus'
}

print(f"\nLabel Map (dari Section 5):")
for k, v in LABEL_MAP.items():
    print(f"  {k} -> {v}")
print(f"\nSentiment Order (prob index -> sentiment):")
for k, v in SENTIMENT_ORDER.items():
    print(f"  prob[{k}] -> {v}")
print()

# ============================================
# 6.3 DATASET CLASS
# ============================================
class ReviewDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return str(self.texts[idx])

# ============================================
# 6.4 PROSES PER FILE CSV
# ============================================
input_folder = './bert_ready_data'
output_folder = './inference_results'
os.makedirs(output_folder, exist_ok=True)

# Cari semua file CSV
csv_files = glob.glob(os.path.join(input_folder, 'bert_clean_*.csv'))

if not csv_files:
    raise ValueError(f"Tidak ada file CSV di {input_folder}!")

print(f"File CSV ditemukan: {len(csv_files)}")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")
print()

# ============================================
# 6.5 FUNGSI INFERENCE PER FILE
# ============================================
def process_single_csv(filepath, batch_size=32):
    """
    Melakukan inference pada satu file CSV dan menyimpan hasilnya
    """
    filename = os.path.basename(filepath)
    product_id = filename.replace('bert_clean_', '').replace('.csv', '')
    
    print(f"\n{'='*60}")
    print(f"PROSES: {filename}")
    print(f"Product ID: {product_id}")
    print(f"{'='*60}")
    
    # Baca CSV
    df = pd.read_csv(filepath)
    print(f"  Baris awal: {len(df)}")
    
    # Validasi kolom
    if 'review' not in df.columns:
        print(f"  ⚠ Kolom 'review' tidak ditemukan! Skip...")
        return None
    
    # Drop missing reviews
    df = df.dropna(subset=['review']).reset_index(drop=True)
    print(f"  Baris setelah drop NA: {len(df)}")
    
    if len(df) == 0:
        print(f"  ⚠ Tidak ada review valid! Skip...")
        return None
    
    # Siapkan dataloader
    texts = df['review'].tolist()
    dataset = ReviewDataset(texts)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    # Inference batch
    all_probs = []
    print(f"  Memulai inference ({len(df)} reviews, batch_size={batch_size})...")
    
    for batch_texts in tqdm(dataloader, desc=f"  {product_id[:30]}"):
        # Tokenize
        inputs = tokenizer(
            batch_texts, 
            return_tensors='pt', 
            truncation=True, 
            padding=True, 
            max_length=512
        )
        # Paksa ke CPU
        inputs = {k: v.cpu() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.extend(probs.cpu().tolist())
    
    print(f"  ✓ Inference selesai! Mendapatkan {len(all_probs)} probability vectors")
    
    # Petakan probabilitas ke sentimen
    s_plus_list = []
    s_netral_list = []
    s_minus_list = []
    predicted_labels = []
    
    for prob in all_probs:
        # Tentukan S_plus, S_netral, S_minus berdasarkan mapping
        sentiment_values = {'S_plus': 0.0, 'S_netral': 0.0, 'S_minus': 0.0}
        
        for idx, sent_key in SENTIMENT_ORDER.items():
            sentiment_values[sent_key] = prob[idx]
        
        s_plus_list.append(sentiment_values['S_plus'])
        s_netral_list.append(sentiment_values['S_netral'])
        s_minus_list.append(sentiment_values['S_minus'])
        
        # Tentukan predicted label (label dengan probabilitas tertinggi)
        max_idx = prob.index(max(prob))
        predicted_label = LABEL_MAP.get(f'LABEL_{max_idx}', f'LABEL_{max_idx}')
        predicted_labels.append(predicted_label)
    
    # Tambahkan ke dataframe
    df_result = df.copy()
    df_result['S_plus'] = s_plus_list
    df_result['S_netral'] = s_netral_list
    df_result['S_minus'] = s_minus_list
    df_result['predicted_label'] = predicted_labels
    
    # Sanity check
    prob_sums = df_result[['S_plus', 'S_netral', 'S_minus']].sum(axis=1)
    invalid_sums = (~prob_sums.between(0.99, 1.01)).sum()
    if invalid_sums > 0:
        print(f"  ⚠ Peringatan: {invalid_sums} baris memiliki jumlah probabilitas ≠ 1.0")
    
    # Statistik prediksi
    print(f"\n  Statistik Prediksi:")
    label_counts = df_result['predicted_label'].value_counts()
    for label, count in label_counts.items():
        pct = count / len(df_result) * 100
        print(f"    {label}: {count} ({pct:.1f}%)")
    
    # Simpan hasil
    output_filename = f"inference_{product_id}.csv"
    output_path = os.path.join(output_folder, output_filename)
    df_result.to_csv(output_path, index=False)
    
    print(f"\n  ✓ Hasil tersimpan: {output_filename}")
    print(f"  Kolom output: {list(df_result.columns)}")
    print(f"  Sample hasil:")
    print(f"    Review: {df_result['review'].iloc[0][:80]}...")
    print(f"    S_plus: {df_result['S_plus'].iloc[0]:.4f}")
    print(f"    S_netral: {df_result['S_netral'].iloc[0]:.4f}")
    print(f"    S_minus: {df_result['S_minus'].iloc[0]:.4f}")
    print(f"    Predicted: {df_result['predicted_label'].iloc[0]}")
    
    return df_result

# ============================================
# 6.6 PROSES SEMUA FILE
# ============================================
print("="*80)
print("MEMULAI INFERENCE PER FILE CSV")
print("="*80)

all_results = {}
total_reviews = 0
success_count = 0
failed_files = []

for filepath in csv_files:
    try:
        result_df = process_single_csv(filepath, batch_size=32)
        if result_df is not None:
            filename = os.path.basename(filepath)
            all_results[filename] = result_df
            total_reviews += len(result_df)
            success_count += 1
    except Exception as e:
        filename = os.path.basename(filepath)
        failed_files.append((filename, str(e)))
        print(f"\n  ✗ GAGAL memproses {filename}: {str(e)}")
        import traceback
        traceback.print_exc()

# ============================================
# 6.7 RINGKASAN AKHIR
# ============================================
print(f"\n{'='*80}")
print(f"RINGKASAN INFERENCE")
print(f"{'='*80}")
print(f"Total file CSV: {len(csv_files)}")
print(f"Berhasil diproses: {success_count}")
print(f"Gagal: {len(failed_files)}")
print(f"Total ulasan diproses: {total_reviews}")

if failed_files:
    print(f"\nFile gagal:")
    for fname, error in failed_files:
        print(f"  - {fname}: {error}")

print(f"\nFile hasil inference:")
print(f"Output folder: {output_folder}")
for f in sorted(os.listdir(output_folder)):
    if f.endswith('.csv'):
        fpath = os.path.join(output_folder, f)
        df_check = pd.read_csv(fpath)
        print(f"  ✓ {f}: {len(df_check)} baris, kolom: {list(df_check.columns)}")

# ============================================
# 6.8 VERIFIKASI CEPAT
# ============================================
print(f"\n{'='*80}")
print(f"VERIFIKASI HASIL INFERENCE")
print(f"{'='*80}")

for f in sorted(os.listdir(output_folder)):
    if f.endswith('.csv'):
        fpath = os.path.join(output_folder, f)
        df_check = pd.read_csv(fpath)
        
        # Cek kolom yang diperlukan
        required_cols = ['S_plus', 'S_netral', 'S_minus', 'predicted_label']
        missing_cols = [c for c in required_cols if c not in df_check.columns]
        
        if missing_cols:
            print(f"  ✗ {f}: KURANG kolom {missing_cols}")
        else:
            # Cek probabilitas
            prob_sums = df_check[['S_plus', 'S_netral', 'S_minus']].sum(axis=1)
            valid_probs = prob_sums.between(0.99, 1.01).sum()
            
            status = "✓ OK" if valid_probs == len(df_check) else "⚠ CEK"
            print(f"  {status} | {f}: {len(df_check)} baris, prob valid: {valid_probs}/{len(df_check)}")

print(f"\n✓ INFERENCE SELESAI!")
print(f"Setiap file CSV memiliki hasil inference-nya sendiri di folder: {output_folder}")
print(f"TIDAK ADA DATA YANG DI-CONCAT!")

SECTION 6: INFERENCE BATCH PER FILE CSV (NO CONCAT)
Using device: cpu

Label Map (dari Section 5):
  LABEL_0 -> POSITIVE
  LABEL_1 -> NEUTRAL
  LABEL_2 -> NEGATIVE

Sentiment Order (prob index -> sentiment):
  prob[0] -> S_plus
  prob[1] -> S_netral
  prob[2] -> S_minus

File CSV ditemukan: 5
  - bert_clean_Temu Toko_Ip 17 Pro.csv
  - bert_clean_PinDuoDuo Y17 Ram.csv
  - bert_clean_TAOB1688_Ip 17 Pro.csv
  - bert_clean_TAOB88.SHOP_ip17.csv
  - bert_clean_Panda_Vivo Y71.csv

MEMULAI INFERENCE PER FILE CSV

PROSES: bert_clean_Temu Toko_Ip 17 Pro.csv
Product ID: Temu Toko_Ip 17 Pro
  Baris awal: 18
  Baris setelah drop NA: 18
  Memulai inference (18 reviews, batch_size=32)...


  Temu Toko_Ip 17 Pro:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Inference selesai! Mendapatkan 18 probability vectors

  Statistik Prediksi:
    POSITIVE: 10 (55.6%)
    NEGATIVE: 5 (27.8%)
    NEUTRAL: 3 (16.7%)

  ✓ Hasil tersimpan: inference_Temu Toko_Ip 17 Pro.csv
  Kolom output: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  Sample hasil:
    Review: keren ini mah hp nya kecil tapi gak kecil baget anak saya suka buat bocill cocok...
    S_plus: 0.9978
    S_netral: 0.0013
    S_minus: 0.0009
    Predicted: POSITIVE

PROSES: bert_clean_PinDuoDuo Y17 Ram.csv
Product ID: PinDuoDuo Y17 Ram
  Baris awal: 103
  Baris setelah drop NA: 103
  Memulai inference (103 reviews, batch_size=32)...


  PinDuoDuo Y17 Ram:   0%|          | 0/4 [00:00<?, ?it/s]

  ✓ Inference selesai! Mendapatkan 103 probability vectors

  Statistik Prediksi:
    POSITIVE: 86 (83.5%)
    NEGATIVE: 11 (10.7%)
    NEUTRAL: 6 (5.8%)

  ✓ Hasil tersimpan: inference_PinDuoDuo Y17 Ram.csv
  Kolom output: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  Sample hasil:
    Review: alhamdulillah udah kedua kalinya order hp di toko ini barang nya bagus dan ori t...
    S_plus: 0.9912
    S_netral: 0.0075
    S_minus: 0.0013
    Predicted: POSITIVE

PROSES: bert_clean_TAOB1688_Ip 17 Pro.csv
Product ID: TAOB1688_Ip 17 Pro
  Baris awal: 3
  Baris setelah drop NA: 3
  Memulai inference (3 reviews, batch_size=32)...


...

  ✓ Inference selesai! Mendapatkan 3 probability vectors

  Statistik Prediksi:
    POSITIVE: 1 (33.3%)
    NEUTRAL: 1 (33.3%)
    NEGATIVE: 1 (33.3%)

  ✓ Hasil tersimpan: inference_TAOB1688_Ip 17 Pro.csv
  Kolom output: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  Sample hasil:
    Review: mantap bos ku syaa mau pesan lagi harga bisa kurang bos...
    S_plus: 0.9638
    S_netral: 0.0289
    S_minus: 0.0073
    Predicted: POSITIVE

PROSES: bert_clean_TAOB88.SHOP_ip17.csv
Product ID: TAOB88.SHOP_ip17
  Baris awal: 25
  Baris setelah drop NA: 25
  Memulai inference (25 reviews, batch_size=32)...


  TAOB88.SHOP_ip17:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Inference selesai! Mendapatkan 25 probability vectors

  Statistik Prediksi:
    POSITIVE: 15 (60.0%)
    NEGATIVE: 7 (28.0%)
    NEUTRAL: 3 (12.0%)

  ✓ Hasil tersimpan: inference_TAOB88.SHOP_ip17.csv
  Kolom output: ['product_id', 'review', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  Sample hasil:
    Review: wort it banget dengan harga segini kualitas juga oke, responsif kamera bagus, da...
    S_plus: 0.9973
    S_netral: 0.0016
    S_minus: 0.0010
    Predicted: POSITIVE

PROSES: bert_clean_Panda_Vivo Y71.csv
Product ID: Panda_Vivo Y71
  Baris awal: 373
  Baris setelah drop NA: 373
  Memulai inference (373 reviews, batch_size=32)...


  Panda_Vivo Y71:   0%|          | 0/12 [00:00<?, ?it/s]

  ✓ Inference selesai! Mendapatkan 373 probability vectors

  Statistik Prediksi:
    POSITIVE: 184 (49.3%)
    NEUTRAL: 120 (32.2%)
    NEGATIVE: 69 (18.5%)

  ✓ Hasil tersimpan: inference_Panda_Vivo Y71.csv
  Kolom output: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  Sample hasil:
    Review: kemudahan penggunaan:...
    S_plus: 0.7237
    S_netral: 0.2733
    S_minus: 0.0030
    Predicted: POSITIVE

RINGKASAN INFERENCE
Total file CSV: 5
Berhasil diproses: 5
Gagal: 0
Total ulasan diproses: 522

File hasil inference:
Output folder: ./inference_results
  ✓ inference_Panda_Vivo Y71.csv: 373 baris, kolom: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  ✓ inference_PinDuoDuo Y17 Ram.csv: 103 baris, kolom: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  ✓ inference_TAOB1688_Ip 17 Pro.csv: 3 baris, kolom: ['product_id', 'review', '

### 7. Pembentukan Sekuens Sentimen per Produk (Strictly Per CSV File)

Membaca file di `with_probs`, menyusun urutan waktunya, dan menyimpannya menjadi file individual di folder `./data/processed/sekuens_per_produk`.

In [ ]:
input_folder = "/Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/inference_results"
output_dir = './sekuens_per_produk'
os.makedirs(output_dir, exist_ok=True)

# ============================================
# DEBUG: CEK ISI FOLDER
# ============================================
print("Mengecek isi folder input...")
print(f"Path: {input_folder}")
print(f"Folder exists: {os.path.exists(input_folder)}")

if os.path.exists(input_folder):
    all_files = os.listdir(input_folder)
    print(f"Semua file di folder ({len(all_files)}):")
    for f in all_files:
        print(f"  - {f}")
    
    csv_files_all = glob.glob(os.path.join(input_folder, '*.csv'))
    print(f"\nFile CSV ditemukan ({len(csv_files_all)}):")
    for f in csv_files_all:
        print(f"  - {os.path.basename(f)}")

# ============================================
# CARI FILE DENGAN PATTERN YANG BENAR
# ============================================
# Coba beberapa pattern
patterns = [
    'inference_*.csv',
    'with_probs_*.csv',
    '*.csv'
]

csv_files = []
for pattern in patterns:
    found = glob.glob(os.path.join(input_folder, pattern))
    if found:
        csv_files = found
        print(f"\n✓ Menggunakan pattern: {pattern} ({len(found)} file)")
        break

if not csv_files:
    print("\n✗ TIDAK ADA FILE CSV DITEMUKAN!")
    print("Pastikan folder inference_results berisi file hasil inference")
else:
    # ============================================
    # PROSES SEKUENS PER FILE
    # ============================================
    print(f"\n{'='*60}")
    print(f"MEMPROSES {len(csv_files)} FILE")
    print(f"{'='*60}")
    
    total_sekuens = 0
    
    for filepath in csv_files:
        filename = os.path.basename(filepath)
        print(f"\nPROSES: {filename}")
        
        df = pd.read_csv(filepath)
        print(f"  Baris: {len(df)}")
        print(f"  Kolom: {list(df.columns)}")
        
        if df.empty:
            print("  ⚠ Dataframe kosong, skip")
            continue
        
        # Konversi timestamp
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            df = df.dropna(subset=['timestamp'])
        else:
            # Jika tidak ada timestamp, buat urutan buatan
            df['timestamp'] = pd.date_range(start='2024-01-01', periods=len(df), freq='h')
        
        # Sort by product_id dan timestamp
        if 'product_id' in df.columns:
            df = df.sort_values(['product_id', 'timestamp'])
        else:
            df['product_id'] = filename.replace('inference_', '').replace('.csv', '')
            df = df.sort_values('timestamp')
        
        # Group by product_id
        if 'product_id' in df.columns:
            grouped = df.groupby('product_id')
        else:
            grouped = [('default', df)]
        
        list_sekuens = []
        
        for product_id, group in grouped:
            # Skip jika T < 3
            if len(group) < 3:
                continue
            
            group_seq = group.copy()
            group_seq['urutan_waktu'] = range(1, len(group) + 1)
            
            # Kolom yang disimpan
            cols_priority = ['product_id', 'label', 'urutan_waktu', 'timestamp', 
                           'S_plus', 'S_netral', 'S_minus']
            cols_present = [c for c in cols_priority if c in group_seq.columns]
            
            # Tambah label jika belum ada
            if 'label' not in group_seq.columns:
                group_seq['label'] = 1
            
            group_seq = group_seq[cols_present]
            list_sekuens.append(group_seq)
            total_sekuens += 1
        
        # Simpan hasil
        if list_sekuens:
            df_sekuens = pd.concat(list_sekuens, ignore_index=True)
            out_name = filename.replace('inference_', 'sekuens_').replace('with_probs_', 'sekuens_')
            out_path = os.path.join(output_dir, out_name)
            df_sekuens.to_csv(out_path, index=False)
            print(f"  ✓ Tersimpan: {out_name} ({len(df_sekuens)} baris, {len(list_sekuens)} sekuens)")
        else:
            print(f"  ⚠ Tidak ada sekuens valid (T >= 3)")
    
    # ============================================
    # VERIFIKASI OUTPUT
    # ============================================
    print(f"\n{'='*60}")
    print(f"VERIFIKASI OUTPUT")
    print(f"{'='*60}")
    
    output_files = glob.glob(os.path.join(output_dir, 'sekuens_*.csv'))
    print(f"File output: {len(output_files)}")
    for f in sorted(output_files):
        df_check = pd.read_csv(f)
        n_products = df_check['product_id'].nunique() if 'product_id' in df_check.columns else 1
        print(f"  ✓ {os.path.basename(f)}: {len(df_check)} baris, {n_products} produk")
    
    print(f"\n✓ SELESAI! Total sekuens: {total_sekuens}")
    print(f"Output folder: {output_dir}")

Mengecek isi folder input...
Path: /Users/asyzyni/Desktop/jam 15 revisi kelarrrrr/inference_results
Folder exists: True
Semua file di folder (5):
  - inference_TAOB1688_Ip 17 Pro.csv
  - inference_PinDuoDuo Y17 Ram.csv
  - inference_TAOB88.SHOP_ip17.csv
  - inference_Panda_Vivo Y71.csv
  - inference_Temu Toko_Ip 17 Pro.csv

File CSV ditemukan (5):
  - inference_TAOB1688_Ip 17 Pro.csv
  - inference_PinDuoDuo Y17 Ram.csv
  - inference_TAOB88.SHOP_ip17.csv
  - inference_Panda_Vivo Y71.csv
  - inference_Temu Toko_Ip 17 Pro.csv

✓ Menggunakan pattern: inference_*.csv (5 file)

MEMPROSES 5 FILE

PROSES: inference_TAOB1688_Ip 17 Pro.csv
  Baris: 3
  Kolom: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  ✓ Tersimpan: sekuens_TAOB1688_Ip 17 Pro.csv (3 baris, 1 sekuens)

PROSES: inference_PinDuoDuo Y17 Ram.csv
  Baris: 103
  Kolom: ['product_id', 'review', 'timestamp', 'label', 'S_plus', 'S_netral', 'S_minus', 'predicted_label']
  ✓ Tersimpan:

In [26]:
import pickle

input_folder = './sekuens_per_produk'
output_folder = './hmm_results'
os.makedirs(output_folder, exist_ok=True)

SEED = 42
np.random.seed(SEED)

csv_files = glob.glob(os.path.join(input_folder, 'sekuens_*.csv'))
if not csv_files:
    raise FileNotFoundError(f"Tidak ada file sekuens di {input_folder}")

# Load semua sekuens
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

df_sekuens = pd.concat(dfs, ignore_index=True)


# Sanity check: S_plus + S_netral + S_minus ≈ 1.0
prob_sum = df_sekuens[['S_plus', 'S_netral', 'S_minus']].sum(axis=1)
invalid = (~prob_sum.between(0.99, 1.01)).sum()
print(f"Baris dengan prob sum ≠ 1.0: {invalid}")

# Cek NaN
nan_count = df_sekuens[['S_plus', 'S_netral', 'S_minus']].isna().sum().sum()
print(f"NaN di kolom prob: {nan_count}")

# Filter T < 3
lengths = df_sekuens.groupby('product_id').size()
valid_products = lengths[lengths >= 3].index
df_sekuens = df_sekuens[df_sekuens['product_id'].isin(valid_products)].copy()

# Distribusi T
print(f"\nDistribusi panjang sekuens (T):")
print(f"  Min: {lengths.min()}, Q1: {lengths.quantile(0.25):.0f}, Median: {lengths.median():.0f}, Q3: {lengths.quantile(0.75):.0f}, Max: {lengths.max()}")


# Susun X dan lengths
product_ids = df_sekuens['product_id'].unique()
X_list = []
lengths_list = []

for pid in product_ids:
    df_prod = df_sekuens[df_sekuens['product_id'] == pid].sort_values('urutan_waktu')
    obs = df_prod[['S_plus', 'S_netral', 'S_minus']].values
    X_list.append(obs)
    lengths_list.append(len(obs))

X = np.concatenate(X_list, axis=0)
lengths = np.array(lengths_list)

print(f"Total observasi: {X.shape[0]}")
print(f"Jumlah produk: {len(lengths)}")

# Training
model = hmm.GaussianHMM(
    n_components=2,
    covariance_type='diag',
    n_iter=100,
    random_state=SEED,
    tol=1e-4,
    verbose=False
)

model.fit(X, lengths)

print(f"Converged: {model.monitor_.converged}")
print(f"Actual iterations: {model.monitor_.iter}")
print(f"Final log-likelihood: {model.score(X, lengths):.2f}")

# Parameter hasil training
print(f"\nTransition matrix:\n{model.transmat_}")
print(f"\nMeans:\nState 0: S_plus={model.means_[0][0]:.4f}, S_netral={model.means_[0][1]:.4f}, S_minus={model.means_[0][2]:.4f}")
print(f"State 1: S_plus={model.means_[1][0]:.4f}, S_netral={model.means_[1][1]:.4f}, S_minus={model.means_[1][2]:.4f}")

# Interpretasi state
if model.means_[0][2] > model.means_[1][2]:
    problematic_state = 0
    stable_state = 1
else:
    problematic_state = 1
    stable_state = 0


temporal_features = []

for pid in tqdm(product_ids, desc="Decoding"):
    df_prod = df_sekuens[df_sekuens['product_id'] == pid].sort_values('urutan_waktu')
    X_prod = df_prod[['S_plus', 'S_netral', 'S_minus']].values
    T = len(X_prod)
    
    # Log-likelihood
    log_lik = model.score(X_prod)
    
    # Viterbi decoding
    state_seq = model.predict(X_prod)
    
    # Proporsi state bermasalah
    prop_problematic = (state_seq == problematic_state).mean()
    
    # Titik transisi paling drastis
    transitions = np.where(np.diff(state_seq) != 0)[0]
    if len(transitions) > 0:
        # Cari transisi dengan perubahan S_minus terbesar
        max_delta = 0
        best_t = transitions[0]
        for t in transitions:
            delta = abs(X_prod[t+1][2] - X_prod[t][2])
            if delta > max_delta:
                max_delta = delta
                best_t = t
        pos_transisi = best_t / T
    else:
        pos_transisi = 1.0
    
    temporal_features.append({
        'product_id': pid,
        'log_lik': log_lik,
        'prop_state_bermasalah': prop_problematic,
        'pos_transisi': pos_transisi
    })

df_temporal = pd.DataFrame(temporal_features)
print(f"Fitur temporal diekstrak untuk {len(df_temporal)} produk")

# ============================================
# SECTION 6: FITUR AGREGAT STATIS
# ============================================
print("\n" + "=" * 60)
print("EKSTRAKSI FITUR AGREGAT STATIS")
print("=" * 60)

df_agregat = df_sekuens.groupby('product_id').agg(
    mean_S_plus=('S_plus', 'mean'),
    std_S_plus=('S_plus', 'std'),
    mean_S_netral=('S_netral', 'mean'),
    std_S_netral=('S_netral', 'std'),
    mean_S_minus=('S_minus', 'mean'),
    std_S_minus=('S_minus', 'std')
).reset_index()

print(f"Fitur agregat diekstrak untuk {len(df_agregat)} produk")

# ============================================
# SECTION 7: KOMBINASI FITUR FINAL
# ============================================
print("\n" + "=" * 60)
print("KOMBINASI FITUR FINAL")
print("=" * 60)

# Ambil label dari data sekuens
df_labels = df_sekuens.groupby('product_id')['label'].first().reset_index()

# Gabungkan semua fitur
df_fitur_final = df_agregat.merge(df_temporal, on='product_id')
df_fitur_final = df_fitur_final.merge(df_labels, on='product_id')

# Urutkan kolom sesuai spesifikasi
kolom_final = [
    'product_id', 'label',
    'mean_S_plus', 'std_S_plus',
    'mean_S_netral', 'std_S_netral',
    'mean_S_minus', 'std_S_minus',
    'log_lik', 'prop_state_bermasalah', 'pos_transisi'
]
df_fitur_final = df_fitur_final[kolom_final]

print(f"Dimensi final: {df_fitur_final.shape}")
print(f"NaN check: {df_fitur_final.isna().sum().sum()}")
print(f"\nSample data:")
print(df_fitur_final.head())

# Simpan
output_path = os.path.join(output_folder, 'fitur_final_per_produk.csv')
df_fitur_final.to_csv(output_path, index=False)

# Simpan model HMM
model_path = os.path.join(output_folder, 'hmm_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)



Baris dengan prob sum ≠ 1.0: 0
NaN di kolom prob: 0

Distribusi panjang sekuens (T):
  Min: 3, Q1: 18, Median: 25, Q3: 103, Max: 373
Total observasi: 522
Jumlah produk: 5
Converged: True
Actual iterations: 18
Final log-likelihood: 1480.21

Transition matrix:
[[0.57394404 0.42605596]
 [0.53231604 0.46768396]]

Means:
State 0: S_plus=0.2142, S_netral=0.4649, S_minus=0.3209
State 1: S_plus=0.9849, S_netral=0.0122, S_minus=0.0029


Decoding:   0%|          | 0/5 [00:00<?, ?it/s]

Fitur temporal diekstrak untuk 5 produk

EKSTRAKSI FITUR AGREGAT STATIS
Fitur agregat diekstrak untuk 5 produk

KOMBINASI FITUR FINAL
Dimensi final: (5, 11)
NaN check: 0

Sample data:
            product_id  label  mean_S_plus  std_S_plus  mean_S_netral  \
0       Panda_Vivo Y71      1     0.492658    0.448783       0.322882   
1    PinDuoDuo Y17 Ram      1     0.801951    0.345198       0.093352   
2   TAOB1688_Ip 17 Pro      1     0.364064    0.520119       0.266306   
3     TAOB88.SHOP_ip17      1     0.579834    0.475046       0.126473   
4  Temu Toko_Ip 17 Pro      1     0.565388    0.495775       0.152879   

   std_S_netral  mean_S_minus  std_S_minus     log_lik  prop_state_bermasalah  \
0      0.407901      0.184460     0.360589  799.166665               0.621984   
1      0.205235      0.104697     0.283971  517.323258               0.330097   
2      0.406902      0.369631     0.492001    5.604935               0.666667   
3      0.286013      0.293693     0.448207   81.02143